In [16]:
from dotenv import load_dotenv
_ = load_dotenv()
import os


In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [12]:
from langchain_core.tools import tool

@tool
def calculate(what: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(what, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {str(e)}"
@tool
def average_dog_weight(name: str) -> str:
    """Returns the average weight of a dog."""
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")  

In [13]:
tools = [calculate, average_dog_weight]
for tool in tools:
    print(type(tool))
    print(tool.name)

<class 'langchain_core.tools.structured.StructuredTool'>
calculate
<class 'langchain_core.tools.structured.StructuredTool'>
average_dog_weight


In [10]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [11]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [26]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns a number - uses Python so be sure to use floating point 
syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given a breed

Example session:
Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A bulldog weighs 51 lbs

You then
output:
Answer: A bullgog weighs 51 lbs.
""".strip()

model = ChatOpenAI(model="gpt-3.5-turbo",
                  api_key=os.environ.get("OPEN_AI_KEY"))
a_bot = Agent(model, tools, system=system_prompt)

In [20]:
messages = [HumanMessage(content="How much does a toy poodle weigh?")]
result = a_bot.graph.invoke({"messages": messages})

Calling: {'name': 'average_dog_weight', 'args': {'name': 'Toy Poodle'}, 'id': 'call_2O3WOa3rZ3GLBsoKXt4mi9FT', 'type': 'tool_call'}
Back to the model!


In [21]:
result

{'messages': [HumanMessage(content='How much does a toy poodle weigh?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 142, 'total_tokens': 160, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DD9PY34uZk9ZkoX7SJ1lQCkt1DapC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9507-589e-7db3-b505-84f61a363339-0', tool_calls=[{'name': 'average_dog_weight', 'args': {'name': 'Toy Poodle'}, 'id': 'call_2O3WOa3rZ3GLBsoKXt4mi9FT', 'type': 'tool_call'}], usage_metadata={'input_tokens': 142, 'output_tokens': 18, 'total_tokens': 160, 'input_token_details': {'au

In [22]:
result['messages'][-1].content

'A Toy Poodle weighs around 7 pounds on average.'

In [27]:
sec_bot = Agent(model, tools, system=system_prompt)

In [28]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

messages = [HumanMessage(content=question)]
result = sec_bot.graph.invoke({"messages": messages})

Calling: {'name': 'average_dog_weight', 'args': {'name': 'Collie'}, 'id': 'call_QtBAAz0UIszWdgwdywiD4hb4', 'type': 'tool_call'}
Calling: {'name': 'average_dog_weight', 'args': {'name': 'Scottish Terrier'}, 'id': 'call_rcYbQ52KNZNUPeonsOumZFxF', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'calculate', 'args': {'what': '37 + 20'}, 'id': 'call_f9gDMrvqoZQyM9nFZZ8hufE8', 'type': 'tool_call'}
Back to the model!


In [29]:
result

{'messages': [HumanMessage(content='I have 2 dogs, a border collie and a scottish terrier. What is their combined weight', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 303, 'total_tokens': 354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DD9USyPEWEQnmvyOpn6MH3oEE8IVI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c950b-ff61-7ae2-a43a-2250e72bc2b9-0', tool_calls=[{'name': 'average_dog_weight', 'args': {'name': 'Collie'}, 'id': 'call_QtBAAz0UIszWdgwdywiD4hb4', 'type': 'tool_call'}, {'name': 'average_dog_weight', 'args': {'name': 'Scottish Te

In [30]:
result['messages'][-1].content

'Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.'